In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
import os
import sys
file_path = os.getcwd()
project_root = os.path.dirname(file_path)
print(project_root)
# Add the project root to the sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
from dotenv import load_dotenv

from langchain.agents import AgentExecutor
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.tools import tool
from typing import Dict, List, Union, Any 
from pathlib import Path
from IPython.display import display, Image
from typing import Optional,Dict, List, Tuple  
# Encoding
import base64
import mimetypes
import io  
import PIL.Image

import subprocess
import os
import sys
import shutil # 
import re
from datetime import datetime 
import os
from pathlib import Path
# Our Libraries
from crunch_llm.code.agents import *
from crunch_llm.code.teams_SciML_PINN import *
from crunch_llm.code.utils import *

/users/jdtoscan/.conda/envs/Agent_Hub/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/oscar/data/gk/jdtoscan/LLM/Crunch_LLM


# Initialization

In [2]:
my_database_path = "/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_pinns"
my_database_path_guides="/users/jdtoscan/data/jdtoscan/LLM/Crunch_LLM/Documents/database_manuals"

In [3]:
naming_agent = StructuredAgent(model='gpt-oss:120b', 
                     verbose=False,
                     system_prompt=naming_agent_system_prompt_PINN,
                    )
retriever_agent = Retriever_Agent(model='gpt-oss:120b', 
                     verbose=False,
                     system_prompt=Retriever_system_prompt_PINN,
                     db_path=my_database_path,
                     collection_name='PINNs_Codes',
                     k=5)

planner_agent = aRAGAgent(model='gemini-2.5-pro', 
                     verbose=False,
                     system_prompt=planner_agent_system_prompt_PINN,
                     model_provider='google',     
                    embeddings_provider='google', 
                    embeddings_model='models/text-embedding-004',
                     db_path=my_database_path_guides,
                     collection_name='PINNs_Manual_v1',
                     k=2)
patcher_agent = StructuredAgent(
                    model='gpt-oss:120b', 
                    system_prompt=patcher_agent_system_prompt_PINN,
                    verbose=False
                )

debugger_agent = StructuredAgent(
                    model='gemini-2.5-flash', 
                     model_provider='google',     
                    system_prompt=debugger_system_prompt_PINN,
                    verbose=False
                )
advisor_agent = aRAGAgent_multimodal(
                    model='gemini-2.5-flash', 
                    verbose=True,
                    system_prompt=advisor_system_prompt_PINN,
                    model_provider='google',     
                    embeddings_provider='google', 
                    embeddings_model='models/text-embedding-004',
                    db_path=my_database_path_guides,
                    collection_name='PINNs_Manual_v1',
                    k=2
                )
advisor_parser = StructuredAgent(
                    model='gpt-oss:120b', 
                     verbose=False,
                     system_prompt=advisor_parser_system_prompt_PINN,
                    )

strategist_agent = aRAGAgent(model='gemini-2.5-flash', 
                     verbose=False,
                     system_prompt=strategist_agent_system_prompt_PINN,
                     model_provider='google',     
                    embeddings_provider='google', 
                    embeddings_model='models/text-embedding-004',
                     db_path=my_database_path_guides,
                     collection_name='PINNs_Manual_v1',
                     k=2)

## User Query

In [4]:
user_question = """
We are solving a forward problem for the Korteweg-De Vries (KdV) equation:

$$
u_t + 6uu_x + u_{xxx} = 0
$$

The spatio-temporal domain is $t \in [0, 5]$ and $x \in [0, 20]$. The boundary conditions are:

$$
u(0,x)=g_0(x) #ICs
u(t,0)=g_1(t) #BCs 1
u(t,20)=g_2(t) #BCs 2
u_x(t,20)=g_3(t) # BCs 3
$$

The analytical solution is given by:

**Parameters:**
* $c_1 = 6.0$
* $c_2 = 2.0$
* $x_1 = -2.0$
* $x_2 = 2.0$

**Helper Variables:**
* $\zeta_1 = x - c_1 \cdot t - x_1$
* $\zeta_2 = x - c_2 \cdot t - x_2$
* $\sqrt{c_1} = \text{jnp.sqrt}(c_1)$
* $\sqrt{c_2} = \text{jnp.sqrt}(c_2)$

**Numerator:**
`num = 2*(c1-c2)*(c1*jnp.cosh(sqrt_c2*zeta2/2)**2 + c2*jnp.sinh(sqrt_c1*zeta1/2)**2)`

**Denominator:**
`term1_den = (sqrt_c1-sqrt_c2)*jnp.cosh((sqrt_c1*zeta1 + sqrt_c2*zeta2)/2)`
`term2_den = (sqrt_c1+sqrt_c2)*jnp.cosh((sqrt_c1*zeta1 - sqrt_c2*zeta2)/2)`
`den = (term1_den + term2_den)**2`
"""

In [5]:
max_iterations_all=20
file_content, filename, retrieval_approval, validator_feedback=retrieval_team_PINN(
                                                                        user_question,
                                                                        retriever_agent,
                                                                      max_iterations= 5)
test_image_summary=False
if retrieval_approval:
    print('We found a suitable code!')    
    refinement_iterations=0
    is_approved = False
    error_report='The code runs perfectly! but it does not satisfy the user requirements.'
    initial_test=''# Pleaceholder for extra instructions
    new_code_details='Test'
    strategist_report=''
    requires_name=True
    debug_approval=True
    while (is_approved == False and refinement_iterations < max_iterations_all):
        print(f"--- Code Creation Iteration: {refinement_iterations} ---")
        file_content,strategist_report= code_refinement_team_PINN(user_question=user_question,
                                            initial_code=file_content,
                                            filename=filename,
                                            error_report=error_report,
                                            strategist_report=strategist_report,
                                            strategist_agent=strategist_agent,
                                            planner_agent=planner_agent,
                                            patcher_agent=patcher_agent,
                                            debug_approval=debug_approval
                                            )  
        project_dir,filename,final_filepath,new_code_details= filing_team_PINN(user_question,
                                                                                filename,
                                                                                file_content,
                                                                                naming_agent,
                                                                                requires_name,
                                                                                new_code_details)
        requires_name=False 
        success,log_file_name=run_code(project_dir,filename,final_filepath)      
        status, debugger_report,debug_approval,full_log_for_agent,log_content=debugging_team_PINN(user_question,
                                                                                        final_filepath,
                                                                                        log_file_name,
                                                                                        debugger_agent)         
        if debug_approval:
            initial_test=''
            error_report, is_approved=advisor_team_PINN(
                                                    user_question,
                                                    project_dir,
                                                    log_content,
                                                    advisor_agent,
                                                    advisor_parser)
        else:
            error_report=f"Debugger Analysis:\n{debugger_report}\n\n"+f"Log File:\n{full_log_for_agent}\n"        
        refinement_iterations=refinement_iterations+1                                                                                                                                   


--- Retrieval Iteration: 0 ---
Retriever Message:
[Allen_Cahn.py]

**Justification**
The `Allen_Cahn.py` template implements a forward PINN using the hybrid Adam + SSBroyden optimizer, which matches the required methodological approach for solving the KdV forward problem with the specified boundary and initial conditions.
✅ Match found: Retriever successfully retrieved 'Allen_Cahn.py'
✅ Retrieval loop approved file: Allen_Cahn.py
We found a suitable code!
--- Code Creation Iteration: 0 ---
--- 1. Cleaning Code ---
--- 2. Calling Strategist Agent ---
--- Strategy Report Received ---
## Scientific Strategy Report

### 1. Determine Problem Nature

*   **Problem Type:** `Forward`
    *   **Reasoning:** The user explicitly states "We are solving a forward problem".
*   **Time Dependence:** `Spatio-Temporal`
    *   **Reasoning:** The problem involves both a time domain ($t \in [0, 5]$) and a spatial domain ($x \in [0, 20]$), and the PDE includes a time derivative ($u_t$).
*   **Data Loss Te